In [1]:
import pandas as pd
import numpy as np
import os
import json
from dotenv import load_dotenv
import requests
import time

In [2]:
import sys
from pathlib import Path

def raiz_proyecto(marcador="src"):
    p = Path.cwd().resolve()
    for candidato in (p, *p.parents):
        if (candidato / marcador).is_dir():
            return candidato
    raise RuntimeError(f"No encuentro '{marcador}' desde {p}")

RAIZ = raiz_proyecto()
DIR_RAW = RAIZ / "data" / "raw"
DIR_PROCESSED = RAIZ / "data" / "processed"
assert DIR_RAW.is_dir() and DIR_PROCESSED.is_dir(), f"Raíz mal resuelta: {RAIZ}"

if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

PATH_PROCESSED = DIR_PROCESSED / "df_final_zonas_temperaturas_2023_2026.parquet"

In [3]:
# 1. Releer el archivo Parquet recién creado
df_t = pd.read_parquet(PATH_PROCESSED, engine="pyarrow")
df_t.columns

Index(['fecha', 'tmin_cantabrico', 'tmin_continental', 'tmin_guadalquivir',
       'tmin_mediterraneo', 'tmax_cantabrico', 'tmax_continental',
       'tmax_guadalquivir', 'tmax_mediterraneo'],
      dtype='str')

In [4]:
fecha_min = df_t['fecha'].min()
fecha_max = df_t["fecha"]. max()
len(pd.date_range(fecha_min, fecha_max))

1294

In [5]:
print(fecha_min, fecha_max)

2023-01-01 00:00:00 2026-07-17 00:00:00


In [6]:
PATH_demanda = DIR_PROCESSED / "demanda_horaria.parquet"
df_d = pd.read_parquet(PATH_demanda, engine="pyarrow")
df_d.columns

Index(['datetime_utc', 'demanda_real', 'demanda_prevista', 'es_evento'], dtype='str')

In [7]:
df_d

,datetime_utc,demanda_real,demanda_prevista,es_evento
0,2023-01-01 00:00:00+00:00,19292.000000,19383.750000,False
1,2023-01-01 01:00:00+00:00,18155.416667,18216.083333,False
2,2023-01-01 02:00:00+00:00,17120.083333,17126.666667,False
3,2023-01-01 03:00:00+00:00,16552.583333,16551.666667,False
4,2023-01-01 04:00:00+00:00,16191.000000,16185.750000,False
...,...,...,...,...
30643,2026-06-30 19:00:00+00:00,37263.250000,37071.666667,False
30644,2026-06-30 20:00:00+00:00,35701.750000,35196.083333,False
30645,2026-06-30 21:00:00+00:00,32987.916667,32756.333333,False
30646,2026-06-30 22:00:00+00:00,30548.416667,30797.333333,False


In [8]:
df_t

,fecha,tmin_cantabrico,tmin_continental,tmin_guadalquivir,tmin_mediterraneo,tmax_cantabrico,tmax_continental,tmax_guadalquivir,tmax_mediterraneo
0,2023-01-01,13.4,4.302901,6.7,7.154945,25.1,15.880851,16.4,17.274176
1,2023-01-02,5.0,7.894004,9.8,7.093681,14.7,12.922244,13.6,16.329670
2,2023-01-03,3.2,2.593037,7.3,8.944505,15.0,13.800967,16.3,18.085714
3,2023-01-04,3.4,-0.671954,8.5,8.086538,17.7,13.380851,18.0,17.175000
4,2023-01-05,5.3,-0.851644,7.8,5.985165,17.7,12.435783,17.6,16.568956
...,...,...,...,...,...,...,...,...,...
1289,2026-07-13,21.5,20.548162,18.2,23.828297,30.7,33.880464,33.1,31.810440
1290,2026-07-14,21.2,17.707930,18.2,24.477198,32.1,37.546809,32.4,32.443681
1291,2026-07-15,21.8,20.387234,18.4,24.526374,30.8,37.068472,36.7,35.211538
1292,2026-07-16,19.4,20.255513,21.6,24.994505,29.5,35.594971,36.1,33.337363


In [10]:
# Construcción de la clave con casteo explícito a datetime64[ns]
df_d['fecha'] = (
    df_d['datetime_utc']
    .dt.tz_convert("Europe/Madrid")
    .dt.normalize()
    .dt.tz_localize(None)
    .astype("datetime64[ns]")
)

# 1. Comprobar dtype exacto
print("Dtype:", df_d['fecha'].dtype)
assert df_d['fecha'].dtype == 'datetime64[ns]', "El tipo de dato debe ser datetime64[ns]"

# 2. Número de fechas únicas
print("Fechas únicas:", df_d['fecha'].nunique())

# 3. Distribución de registros por día
print("\nFrecuencia de horas por día:")
print(df_d['fecha'].value_counts().value_counts())

# 4. Rango de fechas
print("\nFecha mínima:", df_d['fecha'].min())
print("Fecha máxima:", df_d['fecha'].max())

Dtype: datetime64[ns]
Fechas únicas: 1278

Frecuencia de horas por día:
count
24    1269
23       5
25       3
2        1
Name: count, dtype: int64

Fecha mínima: 2023-01-01 00:00:00
Fecha máxima: 2026-07-01 00:00:00


In [11]:
df = df_d.merge(df_t, on="fecha", how="left", validate="m:1")

In [12]:
assert df.shape == (30648, 13), df.shape

# 48 NaN, todos en las dos columnas de guadalquivir
print(df.isna().sum())

# y todos del mismo día
fechas_nan = df.loc[df['tmin_guadalquivir'].isna(), 'fecha'].unique()
print(fechas_nan)

# difusión: un día cualquiera
dia = df[df['fecha'] == '2024-05-14']
print(len(dia), dia['tmin_continental'].nunique())

datetime_utc          0
demanda_real          0
demanda_prevista      0
es_evento             0
fecha                 0
tmin_cantabrico       0
tmin_continental      0
tmin_guadalquivir    24
tmin_mediterraneo     0
tmax_cantabrico       0
tmax_continental      0
tmax_guadalquivir    24
tmax_mediterraneo     0
dtype: int64
<DatetimeArray>
['2023-12-15 00:00:00']
Length: 1, dtype: datetime64[ns]
24 1


In [13]:
df = df[df['fecha'] != '2023-12-15'].reset_index(drop=True)
assert df.shape == (30624, 13), df.shape
assert df.isna().sum().sum() == 0, "Quedan NaN"